<a href="https://colab.research.google.com/github/strongman2026/colab/blob/copilot%2Foptimize-comfyui-colab/colab_comfyui_start_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

首次：1 → 2 → 3 → 6
日常：6
加新节点/模型：4（填配置）→ 5（执行）→ 6（重启）

In [1]:
# ============================================================
# Cell 1 / 6 : 00_env_bootstrap
# 基础环境初始化（首次/重置后运行）
# ============================================================

import os, subprocess, time
from pathlib import Path
from google.colab import drive, userdata

def run(cmd, title="", check=True):
    if title:
        print(f"\n[⏳] {title}")
    print(f"[CMD] {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = p.stdout[-1200:] if p.stdout else "[OK]"
    print(out)
    if check and p.returncode != 0:
        raise RuntimeError(f"命令失败: {cmd}")

def step(i, n, title):
    print(f"\n{'='*64}\n[{i}/{n}] {title}\n{'='*64}")

TOTAL = 7
COMFY_DIR = "/content/ComfyUI"
CUSTOM_NODES_DIR = f"{COMFY_DIR}/custom_nodes"
FRP_DIR = "/content/frp"
FRPC_BIN = f"{FRP_DIR}/frpc"

# 1) 挂载网盘（可选）
step(1, TOTAL, "挂载 Google Drive")
try:
    drive.mount('/content/drive')
except Exception as e:
    print(f"[WARN] Drive 挂载失败（可忽略）: {e}")

# 2) 读取密钥
step(2, TOTAL, "读取 Colab Secrets")
try:
    VPS_IP = userdata.get("VPS_IP")
    FRP_TOKEN = userdata.get("FRP_TOKEN")
    print("✅ 已从 Secrets 读取 VPS_IP / FRP_TOKEN")
except Exception:
    # 兜底（建议你改成自己的，且尽量用 Secrets）
    VPS_IP = "34.153.197.18"
    FRP_TOKEN = "Kaggle2026!"
    print("⚠️ 未从 Secrets 读取到变量，使用代码内兜底值")

# 3) 系统依赖
step(3, TOTAL, "安装系统依赖（git/aria2）")
run("apt-get update -qq && apt-get install -y -qq git aria2", "安装 git + aria2")

# 4) 克隆 ComfyUI
step(4, TOTAL, "克隆 ComfyUI（浅克隆加速）")
if not Path(COMFY_DIR).exists():
    run(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}", "clone ComfyUI")
else:
    print("ℹ️ ComfyUI 已存在，跳过 clone")

# 5) 克隆 Manager
step(5, TOTAL, "克隆 ComfyUI-Manager")
Path(CUSTOM_NODES_DIR).mkdir(parents=True, exist_ok=True)
manager_dir = f"{CUSTOM_NODES_DIR}/ComfyUI-Manager"
if not Path(manager_dir).exists():
    run(f"git clone --depth 1 https://github.com/ltdrdata/ComfyUI-Manager.git {manager_dir}", "clone Manager")
else:
    print("ℹ️ ComfyUI-Manager 已存在，跳过 clone")

# 6) 安装 Python 依赖
step(6, TOTAL, "安装 Python 依赖（无缓存加速）")
run("python -m pip install -q -U pip setuptools wheel", "升级 pip 工具链")
req = f"{COMFY_DIR}/requirements.txt"
if Path(req).exists():
    run(f"python -m pip install -q --no-cache-dir -r {req}", "安装 ComfyUI requirements")
else:
    print("⚠️ requirements.txt 不存在，跳过")

# 7) 安装 FRP & 写配置
step(7, TOTAL, "部署 FRP 客户端并生成配置")
Path(FRP_DIR).mkdir(parents=True, exist_ok=True)
if not Path(FRPC_BIN).exists():
    run(
        f"aria2c -x 16 -s 16 -k 1M -d {FRP_DIR} -o frp.tar.gz "
        "https://github.com/fatedier/frp/releases/download/v0.61.1/frp_0.61.1_linux_amd64.tar.gz",
        "下载 FRP"
    )
    run(f"tar -xzf {FRP_DIR}/frp.tar.gz -C {FRP_DIR} --strip-components=1", "解压 FRP")
    run(f"chmod +x {FRPC_BIN}", "设置执行权限")
else:
    print("ℹ️ frpc 已存在，跳过下载")

frpc_toml = f"""
serverAddr = "{VPS_IP}"
serverPort = 7000
auth.method = "token"
auth.token = "{FRP_TOKEN}"

[[proxies]]
name = "comfyui_colab"
type = "tcp"
localIP = "127.0.0.1"
localPort = 8188
remotePort = 8188
""".strip()

with open(f"{FRP_DIR}/frpc.toml", "w", encoding="utf-8") as f:
    f.write(frpc_toml + "\n")

print("\n✅ Cell 1 完成：基础环境就绪")


[1/7] 挂载 Google Drive
Mounted at /content/drive

[2/7] 读取 Colab Secrets
✅ 已从 Secrets 读取 VPS_IP / FRP_TOKEN

[3/7] 安装系统依赖（git/aria2）

[⏳] 安装 git + aria2
[CMD] apt-get update -qq && apt-get install -y -qq git aria2
ers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link



In [2]:
# ============================================================
# Cell 2 / 6 : 01_model_base_prepare
# 基础模型准备（挂载 + 外链下载）
# ============================================================

import os, glob, subprocess
from pathlib import Path

def run(cmd, title="", check=True):
    if title:
        print(f"\n[⏳] {title}")
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = p.stdout[-1000:] if p.stdout else "[OK]"
    print(out)
    if check and p.returncode != 0:
        raise RuntimeError(f"命令失败: {cmd}")

def bar(prefix, i, n, name=""):
    n = max(n, 1)
    d = int(i/n*24)
    print(f"{prefix} [{'#'*d}{'.'*(24-d)}] {i}/{n} {name}")

COMFY_DIR = "/content/ComfyUI"
MODELS_DIR = f"{COMFY_DIR}/models"
Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)

# 你已有的本地数据集挂载
KAGGLE_DATASETS = [
    ("checkpoints", "/kaggle/input/animagine-xl-4-0-comfyui*/*.safetensors"),
    ("checkpoints", "/kaggle/input/datasets/jarredstroman/my-pony-v6-private/*.safetensors"),
    ("loras", "/kaggle/input/datasets/jarredstroman/family-guy-style/*.safetensors"),
]

# 可选外链
EXTERNAL_URLS = [
    # ("checkpoints", "https://huggingface.co/.../model.safetensors", None),
]

print("[1/3] 创建目录并清理旧软链")
for d in ["checkpoints","loras","unet","vae","clip","clip_vision","controlnet","ipadapter","xlabs/ipadapters"]:
    Path(f"{MODELS_DIR}/{d}").mkdir(parents=True, exist_ok=True)
    run(f"find {MODELS_DIR}/{d} -type l -delete", f"清理旧软链: {d}", check=False)

print("\n[2/3] 挂载 Kaggle 本地模型")
for i, (subdir, pattern) in enumerate(KAGGLE_DATASETS, 1):
    files = glob.glob(pattern)
    target = Path(f"{MODELS_DIR}/{subdir}")
    for src in files:
        dst = target / Path(src).name
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        os.symlink(src, dst)
    bar("挂载进度", i, len(KAGGLE_DATASETS), f"{subdir}: {len(files)} files")

print("\n[3/3] 外链模型下载（可选）")
if EXTERNAL_URLS:
    for i, (subdir, url, fname) in enumerate(EXTERNAL_URLS, 1):
        target = Path(f"{MODELS_DIR}/{subdir}")
        target.mkdir(parents=True, exist_ok=True)
        filename = fname if fname else url.split("?")[0].split("/")[-1]
        run(f"aria2c -c -x 16 -s 16 -k 1M '{url}' -d '{target}' -o '{filename}'", f"下载 {filename}")
        bar("下载进度", i, len(EXTERNAL_URLS), filename)
else:
    print("ℹ️ EXTERNAL_URLS 为空，跳过")

print("\n✅ Cell 2 完成：基础模型就绪")

[1/3] 创建目录并清理旧软链

[⏳] 清理旧软链: checkpoints
[OK]

[⏳] 清理旧软链: loras
[OK]

[⏳] 清理旧软链: unet
[OK]

[⏳] 清理旧软链: vae
[OK]

[⏳] 清理旧软链: clip
[OK]

[⏳] 清理旧软链: clip_vision
[OK]

[⏳] 清理旧软链: controlnet
[OK]

[⏳] 清理旧软链: ipadapter
[OK]

[⏳] 清理旧软链: xlabs/ipadapters
[OK]

[2/3] 挂载 Kaggle 本地模型
挂载进度 [########................] 1/3 checkpoints: 0 files
挂载进度 [################........] 2/3 checkpoints: 0 files
挂载进度 [########################] 3/3 loras: 0 files

[3/3] 外链模型下载（可选）
ℹ️ EXTERNAL_URLS 为空，跳过

✅ Cell 2 完成：基础模型就绪


In [3]:
# ============================================================
# Cell 3 / 6 : 02_plugin_base_install
# 基础插件安装
# ============================================================

import subprocess
from pathlib import Path

def run(cmd, title="", check=True):
    if title:
        print(f"\n[⏳] {title}")
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = p.stdout[-1000:] if p.stdout else "[OK]"
    print(out)
    if check and p.returncode != 0:
        raise RuntimeError(f"命令失败: {cmd}")

COMFY_DIR = "/content/ComfyUI"
CUSTOM_NODES_DIR = f"{COMFY_DIR}/custom_nodes"
Path(CUSTOM_NODES_DIR).mkdir(parents=True, exist_ok=True)

BASE_PLUGINS = [
    "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git",
    "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
    "https://github.com/rgthree/rgthree-comfy.git",
]

for i, url in enumerate(BASE_PLUGINS, 1):
    name = url.rstrip("/").split("/")[-1].replace(".git", "")
    dst = Path(f"{CUSTOM_NODES_DIR}/{name}")
    print(f"\n[{i}/{len(BASE_PLUGINS)}] 插件: {name}")

    if not dst.exists():
        run(f"git clone --depth 1 {url} {dst}", f"克隆 {name}")
    else:
        run(f"cd {dst} && git pull -q", f"更新 {name}", check=False)

    req = dst / "requirements.txt"
    if req.exists():
        run(f"python -m pip install -q --no-cache-dir -r {req}", f"安装依赖 {name}")
    else:
        print("ℹ️ 无 requirements.txt，跳过")

print("\n✅ Cell 3 完成：基础插件就绪")


[1/3] 插件: ComfyUI_IPAdapter_plus

[⏳] 克隆 ComfyUI_IPAdapter_plus
Cloning into '/content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus'...

ℹ️ 无 requirements.txt，跳过

[2/3] 插件: comfyui_controlnet_aux

[⏳] 克隆 comfyui_controlnet_aux
Cloning into '/content/ComfyUI/custom_nodes/comfyui_controlnet_aux'...


[⏳] 安装依赖 comfyui_controlnet_aux
[OK]

[3/3] 插件: rgthree-comfy

[⏳] 克隆 rgthree-comfy
Cloning into '/content/ComfyUI/custom_nodes/rgthree-comfy'...


[⏳] 安装依赖 rgthree-comfy
[OK]

✅ Cell 3 完成：基础插件就绪


In [ ]:
# ============================================================
# Cell 4 / 6 : 03_expansion_center
# 扩展配置中心（仅配置，不执行下载）
# ============================================================

# True: 需要在 ComfyUI-Manager 中安装新节点时打开
# False: 日常极速模式（可隐藏 Manager）
EXPANSION_MODE = True

# 1) 新增节点（Git 仓库）
NEW_CUSTOM_NODES = [
    # "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
]

# 2) 新增模型：URL 直链
# (subdir, url, filename_or_none)
NEW_MODELS_URL = [
    # ("checkpoints", "https://huggingface.co/.../xxx.safetensors", None),
]

# 3) 新增模型：HuggingFace 仓库
# (repo_id, filename, subdir)
NEW_MODELS_HF = [
    # ("XLabs-AI/flux-ip-adapter", "ip_adapter.safetensors", "xlabs/ipadapters"),
]

# 4) 新增模型：Kaggle 本地挂载
# (subdir, glob_pattern)
NEW_MODELS_KAGGLE = [
    # ("loras", "/kaggle/input/your-dataset/*.safetensors"),
]

print("✅ Cell 4 配置已加载")
print(f"- EXPANSION_MODE: {EXPANSION_MODE}")
print(f"- NEW_CUSTOM_NODES: {len(NEW_CUSTOM_NODES)}")
print(f"- NEW_MODELS_URL: {len(NEW_MODELS_URL)}")
print(f"- NEW_MODELS_HF: {len(NEW_MODELS_HF)}")
print(f"- NEW_MODELS_KAGGLE: {len(NEW_MODELS_KAGGLE)}")

In [ ]:
# ============================================================
# Cell 5 / 6 : 04_apply_expansion
# 应用扩展：新节点 + 新模型
# 依赖 Cell 4 里的变量
# ============================================================

import os, glob, subprocess
from pathlib import Path

def run(cmd, title="", check=True):
    if title:
        print(f"\n[⏳] {title}")
    print(f"[CMD] {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = p.stdout[-1200:] if p.stdout else "[OK]"
    print(out)
    if check and p.returncode != 0:
        raise RuntimeError(f"命令失败: {cmd}")

def bar(prefix, i, n, name=""):
    n = max(n, 1)
    d = int(i/n*24)
    print(f"{prefix} [{'#'*d}{'.'*(24-d)}] {i}/{n} {name}")

COMFY_DIR = "/content/ComfyUI"
CUSTOM_NODES_DIR = f"{COMFY_DIR}/custom_nodes"
MODELS_DIR = f"{COMFY_DIR}/models"
Path(CUSTOM_NODES_DIR).mkdir(parents=True, exist_ok=True)
Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)

print("[0/4] 切换 Manager 状态")
manager_dir = Path(f"{CUSTOM_NODES_DIR}/ComfyUI-Manager")
hidden_dir = Path(f"{COMFY_DIR}/ComfyUI-Manager_hidden")
if 'EXPANSION_MODE' not in globals():
    EXPANSION_MODE = True

if EXPANSION_MODE:
    if hidden_dir.exists() and not manager_dir.exists():
        hidden_dir.rename(manager_dir)
    print("✅ Expansion Mode: Manager 已开启")
else:
    if manager_dir.exists() and not hidden_dir.exists():
        manager_dir.rename(hidden_dir)
    print("⚡ Fast Mode: Manager 已隐藏")

print("\n[1/4] 安装新增节点")
NEW_CUSTOM_NODES = globals().get("NEW_CUSTOM_NODES", [])
for i, url in enumerate(NEW_CUSTOM_NODES, 1):
    name = url.rstrip("/").split("/")[-1].replace(".git", "")
    dst = Path(f"{CUSTOM_NODES_DIR}/{name}")
    bar("节点进度", i, len(NEW_CUSTOM_NODES), name)

    if not dst.exists():
        run(f"git clone --depth 1 {url} {dst}", f"克隆 {name}")
    else:
        run(f"cd {dst} && git pull -q", f"更新 {name}", check=False)

    req = dst / "requirements.txt"
    if req.exists():
        run(f"python -m pip install -q --no-cache-dir -r {req}", f"安装依赖 {name}")
if not NEW_CUSTOM_NODES:
    print("ℹ️ 无新增节点")

print("\n[2/4] 下载 URL 模型")
NEW_MODELS_URL = globals().get("NEW_MODELS_URL", [])
for i, (subdir, url, filename) in enumerate(NEW_MODELS_URL, 1):
    target = Path(f"{MODELS_DIR}/{subdir}")
    target.mkdir(parents=True, exist_ok=True)
    fname = filename if filename else url.split("?")[0].split("/")[-1]
    bar("URL模型进度", i, len(NEW_MODELS_URL), fname)
    run(f"aria2c -c -x 16 -s 16 -k 1M '{url}' -d '{target}' -o '{fname}'", f"下载 {fname}")
if not NEW_MODELS_URL:
    print("ℹ️ 无 URL 模型")

print("\n[3/4] 挂载 Kaggle 本地模型")
NEW_MODELS_KAGGLE = globals().get("NEW_MODELS_KAGGLE", [])
for i, (subdir, pattern) in enumerate(NEW_MODELS_KAGGLE, 1):
    target = Path(f"{MODELS_DIR}/{subdir}")
    target.mkdir(parents=True, exist_ok=True)
    files = glob.glob(pattern)
    bar("Kaggle挂载进度", i, len(NEW_MODELS_KAGGLE), f"{subdir}: {len(files)} files")
    for src in files:
        dst = target / Path(src).name
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        os.symlink(src, dst)
if not NEW_MODELS_KAGGLE:
    print("ℹ️ 无 Kaggle 模型")

print("\n[4/4] 下载 HF 模型")
NEW_MODELS_HF = globals().get("NEW_MODELS_HF", [])
if NEW_MODELS_HF:
    run("python -m pip install -q -U huggingface_hub", "安装 huggingface_hub")
    from huggingface_hub import hf_hub_download
    Path("/content/dataset").mkdir(parents=True, exist_ok=True)

    for i, (repo_id, filename, subdir) in enumerate(NEW_MODELS_HF, 1):
        bar("HF模型进度", i, len(NEW_MODELS_HF), filename)
        local = hf_hub_download(repo_id=repo_id, filename=filename, local_dir="/content/dataset")
        target = Path(f"{MODELS_DIR}/{subdir}")
        target.mkdir(parents=True, exist_ok=True)
        dst = target / Path(filename).name
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        os.symlink(local, dst)
else:
    print("ℹ️ 无 HF 模型")

print("\n✅ Cell 5 完成：扩展已应用")
print("⚠️ 建议执行 Cell 6 重启运行时服务")

In [ ]:
# ============================================================
# Cell 6 / 6 : 05_start_runtime_only
# 仅启动 ComfyUI + FRP（日常只跑这个）
# ============================================================

import os, time, socket, subprocess
from pathlib import Path

COMFY_DIR = "/content/ComfyUI"
FRP_DIR = "/content/frp"
FRPC_BIN = f"{FRP_DIR}/frpc"

# 日常建议 False（更快）；需要 Manager 装新节点时 True
ENABLE_MANAGER = False

# 这里读取 frpc.toml 的 serverAddr 更稳
VPS_IP = "34.153.197.18"

def wait_port(host, port, timeout=120):
    start = time.time()
    while time.time() - start < timeout:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex((host, port)) == 0:
                return True
        time.sleep(1)
    return False

print("[1/5] 清理旧进程")
os.system("pkill -f 'python.*main.py' || true")
os.system("pkill -f frpc || true")
time.sleep(1)

print("[2/5] 切换 Manager 开关")
manager_dir = Path(f"{COMFY_DIR}/custom_nodes/ComfyUI-Manager")
hidden_dir = Path(f"{COMFY_DIR}/ComfyUI-Manager_hidden")
if ENABLE_MANAGER:
    if hidden_dir.exists() and not manager_dir.exists():
        hidden_dir.rename(manager_dir)
else:
    if manager_dir.exists() and not hidden_dir.exists():
        manager_dir.rename(hidden_dir)

print("[3/5] 启动 ComfyUI")
comfy_proc = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188", "--normalvram"],
    cwd=COMFY_DIR
)

print("[4/5] 等待 8188 端口")
if not wait_port("127.0.0.1", 8188, timeout=120):
    raise RuntimeError("ComfyUI 启动超时，请检查日志")

print("[5/5] 启动 FRP")
if not Path(FRPC_BIN).exists():
    raise FileNotFoundError(f"未找到 FRPC: {FRPC_BIN}，请先运行 Cell 1")
frp_proc = subprocess.Popen([FRPC_BIN, "-c", f"{FRP_DIR}/frpc.toml"])

print("\n" + "="*58)
print("✅ ComfyUI 已启动")
print(f"🌐 访问地址: http://{VPS_IP}:8188")
print(f"🧩 Manager: {'开启' if ENABLE_MANAGER else '隐藏'}")
print("="*58 + "\n")

comfy_proc.wait()

[1/5] 清理旧进程
[2/5] 切换 Manager 开关
[3/5] 启动 ComfyUI
[4/5] 等待 8188 端口
[5/5] 启动 FRP

✅ ComfyUI 已启动
🌐 访问地址: http://34.153.197.18:8188
🧩 Manager: 隐藏

